# CS336 Spring 2026 - Special Tokens
----
> Special-token 应该是整个字符串是一个 token, 不能让普通 regex 或 BPE 决定怎么拆.
> 
> 本节最重要的思想是: Special-token 检测是在 pre-tokenization 之前.

## 1. 先恢复上一节的 pre-tokenizer

In [1]:
import regex

GPT2_PRETOKEN_PATTERN = (
    r"""'(?:[sdmt]|ll|ve|re)"""
    r"""| ?\p{L}+"""
    r"""| ?\p{N}+"""
    r"""| ?[^\s\p{L}\p{N}]+"""
    r"""|\s+(?!\S)"""
    r"""|\s+"""
)


def pretokenize(text: str) -> list[str]:

    return [match.group(0) for match in regex.finditer(GPT2_PRETOKEN_PATTERN, text)]


## 2. 错误实验: 将 Special Token 作为普通文本
本来希望 `<|endoftext|>` 是一个完整的 token, 但是普通的 pre-tokenizer 将其拆开. 若继续使用 BPE, 这些会继续进入下一部分, 但显然是不对的.

In [2]:
text = "Hello<|endoftext|>world"

print(pretokenize(text))

['Hello', '<|', 'endoftext', '|>', 'world']


## 3. Special Token 的核心性质: Atomic
若定义了:
```python
special_tokens = ["<|endoftext|>"]
```
那么下面所有的位置:
```text
<|endoftext|>

hello<|endoftext|>

<|endoftext|>hello

hello<|endoftext|>world
```
都必须把 `<|endoftext|>` 识别成一个整体. 这就是所谓的 atomic token.
> tokenizer 的普通逻辑不能继续深入 special token 内部.

## 4. 正确的处理顺序
正确顺序为:
```text
whole text
    ↓
先检测 special tokens
    ↓
切分成 special spans / ordinary spans
    ↓
只有 ordinary spans 进入 regex
```
即 `Hello<|endoftext|>world`, 先切:
```text
ORDINARY  "Hello"

SPECIAL   "<|endoftext|>"

ORDINARY  "world"
```
然后只有 `ORDERINARY` 进入普通的 pre-tokenization.

## 5. 构造 Special Token Pattern
我们需要一个 pattern 来寻找 special tokens. 先考虑:
```python
special_tokens = ["<|endoftext|>"]
```
注意这里的 `< | >` 在 regex 中具有特殊意义. 因此需要:
```python
regex.escape(...)
```

In [3]:
def build_special_pattern(special_tokens: list[str]) -> regex.Pattern:
    # 构造 Special Token Pattern
    if not special_tokens:
        # 注意: 这里不能 return None, 否则返回类型会变成 Pattern | None,
        # 下游的 pattern.pattern / pattern.split 就会报 "pattern" 不是 "None" 的已知属性.
        return regex.compile(r"(?!x)x")  # 永不匹配的哨兵 pattern

    ordered = sorted(
        set(special_tokens), key=lambda token: (-len(token), token)
    )  # 去重并按长度降序排序
    alternatives = "|".join(regex.escape(token) for token in ordered)

    return regex.compile(f"({alternatives})")

### Q: 为什么上述要进行长度降序排序?
A: 正则中的`|` 通常从左到右匹配, 一旦匹配成功就停止. 如果按较短的 token 排在前面, 它可能会先匹配, 导致较长的 token 被错误的拆开.

 例如 `<|endof` 和 `<|endoftext|>`, 若 `<|endof` 在前, 文本 `<|endoftext|>` 会被拆成 `<endof` + `text|>`.

In [4]:
pattern = build_special_pattern(["<|endoftext|>"])

print(pattern.pattern)

(<\|endoftext\|>)


可以看到 `|` 被 escape 成了 `\|`

### Q: 为什么使用了 capturing group?
A: 因为是在后续希望 `regex.split(...)` 不仅返回 special token 两边的文本, 还把 special token 本身保存下来.

例如, 对 `Hello<|endoftext|>World`, 希望
```text
Hello

<|endoftext|>

World
```
而不是将 special token 删除掉.

## 6. 隔离 Ordinary Span 和 Special Span

In [5]:
def split_around_special_tokens(
    text: str, special_tokens: list[str]
) -> list[tuple[bool, str]]:
    pattern = build_special_pattern(special_tokens)

    special_set = set(special_tokens)
    output: list[tuple[bool, str]] = []

    for piece in pattern.split(text):
        # split 会在相邻/首尾 special token 处产生空字符串[], 必须丢弃
        if piece == "":
            continue

        output.append((piece in special_set, piece))

    return output

In [6]:
# 进行测试
print(split_around_special_tokens("Hello<|endoftext|>world", ["<|endoftext|>"]))

[(False, 'Hello'), (True, '<|endoftext|>'), (False, 'world')]


## 7. Special-token Split 也必须满足 Round Trip
和上一节 pre-tokenization 一样, 不能损失任何字符. 所以另一个 invariant 是:

<center>
join(split_special(text))=text
</center>

In [7]:
# 进行测试
text = "A<|endoftext|>B<|endoftext|>C"

pieces = split_around_special_tokens(text, ["<|endoftext|>"])
print("分隔后的字符 = ", pieces)

reconstructed = "".join(piece for _, piece in pieces)
print("重构后的字符串 = ", reconstructed)

print("重构后的字符串与原始字符串是否相同 = ", reconstructed == text)

分隔后的字符 =  [(False, 'A'), (True, '<|endoftext|>'), (False, 'B'), (True, '<|endoftext|>'), (False, 'C')]
重构后的字符串 =  A<|endoftext|>B<|endoftext|>C
重构后的字符串与原始字符串是否相同 =  True


## 8. 连续的 Special tokens 测试

In [8]:
text = "hello<|endoftext|><|endoftext|>world"

pieces = split_around_special_tokens(text, ["<|endoftext|>"])

for is_special, piece in pieces:
    print(("SPECIAL " if is_special else "ORDINARY"), repr(piece))


ORDINARY 'hello'
SPECIAL  '<|endoftext|>'
SPECIAL  '<|endoftext|>'
ORDINARY 'world'


这里不能将 `<|endoftext|><|endoftext|>` 当作一般的 ordinary string, 然后再进行 pre-tokenize, 否则 atomic property 就会失效.

## 9. 更困难的情况: Overlapping Special Tokens
现在考虑:
```python
short = "<|endoftext|>"
long = "<|endoftext|>" "<|endoftext|>"
```
并假设:
```python
special_tokens = [short, long]
```
所以现在的 `long` 也被配置成了一个合法的 special token. 于是输入 `long` 应该匹配 1 个 token, 而不是 2 个. 这就是

<center>
Longest Match
</center>

In [9]:
# 测试 Overlapping Special Tokens
short = "<|endoftext|>"
long = short + short
special_tokens = [short, long]

text = f"Hello{long}world{short}"
pieces = split_around_special_tokens(text, special_tokens)

for is_special, piece in pieces:
    print("SPECIAL" if is_special else "REGULAR", repr(piece))

REGULAR 'Hello'
SPECIAL '<|endoftext|><|endoftext|>'
REGULAR 'world'
SPECIAL '<|endoftext|>'


### Q: 回头看 `build_special_pattern` 的构造, 为什么需要 `key=-len(token)`?
A: 核心就是让 long special token 排在 short special token 前面. 因为 regex alternatives 一般采用 `leftmost-first` 匹配. 这样就不会出现匹配了 short special token, 但是没有匹配 long special token 的情况.


In [10]:
# 错误版本
short = "<|endoftext|>"
long = short + short

naive_pattern = regex.compile(
    f"({regex.escape(short)}|{regex.escape(long)})"
)  # 设置 long 排在 short 后面
correct_pattern = build_special_pattern([short, long])

print("naive:")
print([p for p in naive_pattern.split(long) if p])

print("\nlongest-first:")
print([p for p in correct_pattern.split(long) if p])

naive:
['<|endoftext|>', '<|endoftext|>']

longest-first:
['<|endoftext|><|endoftext|>']


## 10. 将 Special Token 和 Pre-tokenization 结合
目前已经实现了两个组件 `split_around_special_tokens()` 和 `pretokenize()`. 正确的顺序应该是:
```text
raw text
    ↓
split special tokens
    ↓
SPECIAL: 不再拆; ORDINARY: pretokenize
```


In [11]:
# 定义一个辅助函数
def frontend_pieces(text: str, special_tokens: list[str]) -> list[tuple[str, str]]:
    output: list[tuple[str, str]] = []

    for is_special, span in split_around_special_tokens(text, special_tokens):
        if is_special:
            output.append(("SPECIAL", span))
        else:
            for piece in pretokenize(span):
                output.append(("ORDINARY", piece))

    return output

In [12]:
text = "Hello, world<|endoftext|>I'm learning CS336."

for kind, piece in frontend_pieces(text, ["<|endoftext|>"]):
    print(kind.ljust(8), repr(piece))

ORDINARY 'Hello'
ORDINARY ','
ORDINARY ' world'
SPECIAL  '<|endoftext|>'
ORDINARY 'I'
ORDINARY "'m"
ORDINARY ' learning'
ORDINARY ' CS'
ORDINARY '336'
ORDINARY '.'


上述就是想要的 tokenizer 前端结构. `<|endoftext|>` 完全没有进入 GPT-2 regex.

## 11. Encoding 的完整结构
现在可以绘制 `Tokenizer.encode()` 的高层流程为:
```text
                     text
                      │
                      ▼
          special-token detection
                      │
            ┌─────────┴─────────┐
            │                   │
            ▼                   ▼
         SPECIAL             ORDINARY
            │                   │
            │                   ▼
            │             pre-tokenization
            │                   │
            │                   ▼
            │               UTF-8 bytes
            │                   │
            │                   ▼
            │              learned BPE
            │                   │
            ▼                   ▼
      special token ID       token IDs
            │                   │
            └─────────┬─────────┘
                      ▼
                  final IDs
```
由此也可以看到 SPECIAL 的路线很短, 不需要其进入 BPE 部分.

## 12. Special Token 进入 Vocabulary
例如 `<|endoftext|>` 虽然在 UTF-8 编码后有很多 bytes, 但是 vocabulary 可以直接建立：
```text
special_token_id
    ↓
b"<|endoftext|>"
```
所以

<center>
token IDs -> bytes
</center>

这个统一的 invariant 完全没有改变.

In [13]:
special = "<|endoftext|>"
special_bytes = special.encode("utf-8")

print(special_bytes)
print(list(special_bytes))

b'<|endoftext|>'
[60, 124, 101, 110, 100, 111, 102, 116, 101, 120, 116, 124, 62]


## 13. Special Token 不是"特殊的数据类型"
对于 token:
- 普通的 BPE token: `256 -> b"th"` `257 -> b"the"`
- Special token: `50256 -> b"<|endoftext|>"`
对 vocabulary 来说, 本质上都是 `dict[int, bytes]`. 所以 ordinary token 和 special token 最终都可以统一为:

<center>
token IDs -> bytes
</center>

"特殊"主要体现在:
- encode 时如何识别
- training 时不能让普通的 BPE 学习

## 14. 一个重要结论: Decode 可以很统一
在后续实现 `decode(ids)` 时, 可能依然保持这个结构:
```text
token IDs
    ↓
vocab[id]
    ↓
bytes fragments
    ↓
b"".join(...)
    ↓
UTF-8 decode
    ↓
string
```
> Special token 的特殊处理主要在 encoding/training frontend, 而不一定在 decode.

## 15. 再次强调: Training 和 Encoding 必须分开考虑
**对于 Encoding**, 输入 `hello<|endoftext|>world`, 应该有:
```text
hello
↓
ordinary pretokenization + BPE
↓
ordinary IDs


<|endoftext|>
↓
direct lookup
↓
SPECIAL_ID


world
↓
ordinary pretokenization + BPE
↓
ordinary IDs
```
最终结果为:
```text
[
    ...ordinary IDs...,
    SPECIAL_ID,
    ...ordinary IDs...
]
```

**对于 BPE Training**, 希望 `<|endoftext|>` 最终存在于 `vocab` 中
> corpus 中`<|endoftext|>` 的字符不能帮助普通 BPE 学习 merge.
因此:
```text
training corpus
        ↓
isolate special tokens
        ↓
┌──────────────────────┐
│ SPECIAL spans        │
│                      │
│ 不参与普通 BPE统计   │
└──────────────────────┘

ORDINARY spans
        ↓
pre-tokenization
        ↓
pair counting
        ↓
BPE learning
```

In [14]:
text = "hello<|endoftext|>hello"

print("WRONG: ordinary pre-tokenization over whole text")
print(pretokenize(text))

WRONG: ordinary pre-tokenization over whole text
['hello', '<|', 'endoftext', '|>', 'hello']


普通的 BPE trainer 会看到 `<|`, `endoftext`, `|>`等特殊符号. 更糟糕的是, 若边界处理错误, 还可能与周围 ordinary text 发生 interaction. 这就是

<center>
vocabulary contamination
</center>

In [15]:
print("RIGHT: isolate special token first")
print(frontend_pieces(text, ["<|endoftext|>"]))

RIGHT: isolate special token first
[('ORDINARY', 'hello'), ('SPECIAL', '<|endoftext|>'), ('ORDINARY', 'hello')]


## 16. 现在考虑 `pretoken -> frequency`
上一节已经建立了:
```text
ordinary corpus
    ↓
pretokenize
    ↓
pretoken bytes
    ↓
Counter
```
现在只需要在其前面增加:
```text
special-token isolation
```

In [16]:
from collections import Counter


def count_training_pretokens(
    text: str, special_tokens: list[str]
) -> Counter[tuple[int, ...]]:
    counts: Counter[tuple[int, ...]] = Counter()

    for is_special, span in split_around_special_tokens(text, special_tokens):
        if is_special:
            continue  # Special token 跳过 BPE

        for piece in pretokenize(span):
            counts[tuple(piece.encode("utf-8"))] += 1

    return counts

In [17]:
corpus = "the cat<|endoftext|>the cat"
counts = count_training_pretokens(corpus, ["<|endoftext|>"])

for byte_seq, frequency in counts.items():
    print(repr(bytes(byte_seq).decode("utf-8")), "->", frequency)


'the' -> 2
' cat' -> 2


上述输出结果没有 `<|`, `endoftext`, `|>` 等特殊符号, 这正是后续所需要的.

## 17. Special Token 如何进入 Vocabulary
在此注意 BPE pair statistics 和 final Vocabulary. 上述所说的 special token 不参与 BPE pair statistics, 并不是说 special token 不进入 vocab. 最终需要:
```text
vocab:

0      -> b"\x00"
...
255    -> b"\xff"

256+   -> learned BPE tokens
...

special_id
       -> b"<|endoftext|>"
```
因此以后训练循环中需要考虑

<center>
number of BPE merges 和 number of special tokens.
</center>

共同占用最终 vocabulary budget.

## 18. Special Tokens 和 换行符
考虑
```python
text = (
    "hello"
    "<|endoftext|>"
    "\n\n"
    "world"
)
```
这里的 `<|endoftext|>` 是 special token. 但是 `\n\n` 不是, 其仍属于 ordinary text. 不能因为 special-token split 就将其删掉.

In [18]:
text = "hello<|endoftext|>\n\nworld"

for kind, piece in frontend_pieces(text, ["<|endoftext|>"]):
    print(kind.ljust(8), repr(piece))


ORDINARY 'hello'
SPECIAL  '<|endoftext|>'
ORDINARY '\n'
ORDINARY '\n'
ORDINARY 'world'


In [19]:
reconstructed = "".join(piece for _, piece in frontend_pieces(text, ["<|endoftext|>"]))

print("\nround trip:", reconstructed == text)


round trip: True


### Q: 为什么需要注意这个细节
A: 因为上一节的 `regex` 本身就有 `whitespace behavior`. 若先 pre-tokenize 整个文本, 再寻找 special token, 会发生 special token 周围的 whitespace 可能已经和错误的上下文发生 interaction.

正确的方法是: 先把 special token 作为硬边界切开, 然后左右 ordinary spans, 各自正常的 pre-tokenize.

## 19. 三个不同的 Boundary
1. 第一层: **Special-token Boundary**

例如 `hello | <|endoftext|> | word`. 这是最强的边界. 普通的 tokenizer 逻辑不允许跨越.

2. 第二层: **Pre-token Boundary**

例如 `"the" | "cat"`. 对于 BPE 来说, 不允许跨 `|`, 但两个部分仍然都属于 ordinary text.

3. 第三层: **BPE 内部 Token Boundary**

例如 `t | h | e`, 这些 boundary 可以被合并.

## 20. Encoding 的三层逻辑
后续正式实现:
```python
Tokenizer.encode(text)
```
考虑三层逻辑:
```text
Layer 1
Special-token segmentation

        ↓

Layer 2
Ordinary pre-tokenization

        ↓

Layer 3
BPE inside each ordinary pre-token

        ↓

token IDs
```
即

<center>
encode = special segmentation + pretokenization + BPE
</center>

## 21. 本节最后检查

In [20]:
# Basic preservation
SPECIAL = "<|endoftext|>"

pieces = split_around_special_tokens(f"a{SPECIAL}b", [SPECIAL])

assert pieces == [(False, "a"), (True, SPECIAL), (False, "b")]

In [21]:
# Round Trip
text = f"A{SPECIAL}B{SPECIAL}C"

pieces = split_around_special_tokens(text, [SPECIAL])

assert "".join(piece for _, piece in pieces) == text

In [22]:
# Consecutive Specials
text = f"a{SPECIAL}{SPECIAL}b"

pieces = split_around_special_tokens(text, [SPECIAL])

assert [piece for is_special, piece in pieces if is_special] == [SPECIAL, SPECIAL]

In [23]:
# Overlapping Special
LONG_SPECIAL = SPECIAL + SPECIAL

text = f"a{LONG_SPECIAL}b{SPECIAL}"
pieces = split_around_special_tokens(text, [SPECIAL, LONG_SPECIAL])
special_pieces = [piece for is_special, piece in pieces if is_special]

assert special_pieces == [LONG_SPECIAL, SPECIAL]

In [ ]:
# Training Statistics
counts = count_training_pretokens(f"hello{SPECIAL}hello", [SPECIAL])

for byte_seq in counts:
    assert b"<|" not in bytes(byte_seq)

In [25]:
# Newline Preservation
text = f"hello{SPECIAL}\n\nworld"

frontend = frontend_pieces(text, [SPECIAL])

assert "".join(piece for _, piece in frontend) == text

In [26]:
print("All special-token sanity checks passed.")

All special-token sanity checks passed.
